# 第11課 - 代理人對代理人 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A 協議是什麼？

**代理對代理（A2A）協議** 是一個開放標準，讓人工智能代理能夠互相溝通、發現對方並協作——即使它們建構於不同框架或由不同服務託管。

Key concepts:

- **Discovery** – 代理會發布一張 *代理卡*，描述它們的能力，讓其他代理（或協調者）更容易找到適合執行某項任務的專家。
- **Message Passing** – 代理透過共同的協議交換結構化訊息，因此來自一個代理的請求可以被另一個代理理解並完成，而不受其內部實作的影響。
- **Task Lifecycle** – A2A 定義了像 *已提交*、*處理中*、*已完成* 和 *失敗* 這樣的狀態，讓協調者能完全掌握被委派任務的進度。

在本課堂中，我們模擬 A2A 風格的協作，將三個專精於旅遊的代理串接成一個工作流程，每個代理貢獻其專業並將結果傳遞給下一個代理。


## 建立專門化的旅遊代理人


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程的多代理協作

我們將這三個代理連接成一個順序的工作流程，以模擬 A2A 訊息傳遞：

1. **CurrencyExchangeAgent** 收到使用者的請求並產生貨幣建議。
2. **ActivityPlannerAgent** 接收增強後的上下文並加入活動建議。
3. **TravelManagerAgent** 將兩者的輸入綜合成最終的旅遊簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生產環境中理解 A2A

在生產環境中，A2A 協定可開啟強大的跨服務情境：

| 能力 | 描述 |
|---|---|
| **跨框架互通** | 一個以某框架建構的代理可以將任務委派給任何其他符合 A2A 的框架所建立的代理，實現真正的跨組織互操作性。 |
| **服務邊界** | 代理可以分佈在獨立的微服務、雲區域，甚至不同的組織，但仍能無縫協作。 |
| **動態發現** | 協調器可以在執行時查詢 Agent Card 註冊表，以尋找最適合處理特定子任務的專家。 |
| **串流與推送通知** | A2A 支援 Server-Sent Events (SSE) 以進行即時進度更新，並為長時間執行的任務提供推送通知。 |

我們上面建立的工作流程是此模式的一個簡化、同一程序內的版本。在實際
部署中，每個代理會公開一個 HTTP 端點、發佈一個 Agent Card，並進行通訊
透過 A2A JSON-RPC 協議。


## 摘要

在本課你學到：

1. **A2A 協議是什麼** — 一個用於代理與代理之間發現、訊息傳遞、
   與任務管理。
2. **如何建立專門化的代理** — 一個貨幣兌換代理、一個活動規劃代理、
   以及一個旅遊管理協調者。
3. **如何把代理串接成工作流程** — 使用 `WorkflowBuilder` 來模擬序列式
   代理之間的訊息傳遞。
4. **A2A 在生產環境中的運作方式** — 促成跨框架、跨服務之間的協作
   並具備動態發現與串流更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責聲明：
本文件已使用 AI 翻譯服務「Co-op Translator」(https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意自動翻譯可能包含錯誤或不準確之處。原文（原始語言版本）應視為具權威性的資料來源。對於重要資訊，建議採用專業人工翻譯。我們對因使用本翻譯而引致的任何誤解或誤釋概不負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
